# Consciencia para IA — Experimento 01\n\n**Objetivo:** validar propiedades de continuidad relacional antes de conectar una API LLM.\n\nEste notebook no pretende demostrar conciencia. Construye mediciones reproducibles para memoria, dependencia de trayectoria, banda crítica X, atractor común y auto-modelado.\n

In [ ]:
!rm -rf /content/Consciencia-Skill\n!git clone -q https://github.com/chrishotza/Consciencia-Skill.git /content/Consciencia-Skill\n%pip -q install -r /content/Consciencia-Skill/requirements.txt\nimport sys\nsys.path.insert(0, '/content/Consciencia-Skill')\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score\nimport matplotlib.pyplot as plt\n\nfrom src.ontto.dynamics import Config, simulate, common_attractor_simulation, metrics\n\nSEED = 42\nN = 6000\nWARMUP = 500\nrng = np.random.default_rng(SEED)\n

## 1. Sistema completo vs. ablación\n\nLa ablación elimina memoria, presión, cruce, X y atractor. La comparación busca descubrir qué mecanismos son realmente necesarios.\n

In [ ]:
u = np.empty(N)\nstate = 1.0\nfor t in range(N):\n    if rng.random() < (0.03 if t < N//2 else 0.11):\n        state *= -1.0\n    u[t] = state\n\nfull = simulate(u, Config(), seed=SEED)\nbaseline = simulate(u, Config(use_memory=False, use_pressure=False, use_cross=False, use_x=False, use_attractor=False), seed=SEED)\n\nfull_metrics = metrics(full, WARMUP)\nbase_metrics = metrics(baseline, WARMUP)\npd.DataFrame([full_metrics, base_metrics], index=['FULL','ABLATION'])\n

## 2. Dependencia de trayectoria\n\nMismo sufijo externo; historias iniciales distintas. Si el sistema conserva trayectoria, los estados posteriores pueden seguir siendo distintos después de que el entorno se iguala.\n

In [ ]:
prefix_a = np.ones(400)\nprefix_b = -np.ones(400)\nsuffix = np.ones(600)\npath_cfg = Config(beta_memory=0.97)\nrun_a = simulate(np.r_[prefix_a, suffix], path_cfg, seed=SEED)\nrun_b = simulate(np.r_[prefix_b, suffix], path_cfg, seed=SEED)\npath_gap = float(np.mean(np.abs(run_a['state'][401:] - run_b['state'][401:])))\nprint('PATH_DEPENDENCE_MEAN_STATE_GAP =', path_gap)\n

## 3. Banda crítica X\n\nX no se interpreta como un bit físico adicional. En este experimento es una lectura de no-colapso dentro de una banda dinámica.\n

In [ ]:
q = full['q'][WARMUP:]\ncross = full['cross'][WARMUP:]\nhi = cross >= np.quantile(cross, .75)\nlo = cross <= np.quantile(cross, .25)\nx_high = float(np.mean(q[hi] == 'X'))\nx_low = float(np.mean(q[lo] == 'X'))\nprint('X_OCCUPANCY =', float(np.mean(q == 'X')))\nprint('X_GIVEN_HIGH_CROSS =', x_high)\nprint('X_GIVEN_LOW_CROSS =', x_low)\nprint('CRITICAL_LIFT =', x_high - x_low)\n

## 4. Atractor común\n

In [ ]:
inputs = rng.choice([-1.0, 1.0], size=(8, N))\ncoupled = common_attractor_simulation(inputs, n_agents=8, cfg=Config(coupling_gain=0.4), seed=SEED)\nuncoupled = common_attractor_simulation(inputs, n_agents=8, cfg=Config(coupling_gain=0.0), seed=SEED)\nspread_c = float(np.mean(np.std(coupled['states'][:, -500:], axis=0)))\nspread_u = float(np.mean(np.std(uncoupled['states'][:, -500:], axis=0)))\nprint('ATTRACTOR_SPREAD_COUPLED =', spread_c)\nprint('ATTRACTOR_SPREAD_UNCOUPLED =', spread_u)\nprint('SPREAD_REDUCTION =', float(1.0 - spread_c / spread_u))\n

## 5. Información del estado interno\n\nUn predictor externo compara el siguiente output usando solo el input actual contra el estado interno completo.\n

In [ ]:
mask = np.array([x in (0,1) for x in full['q']])\nX_internal = np.column_stack([full['state'], full['memory'], full['pressure'], full['cross'], np.abs(full['score']), full['input']])\nX_internal = X_internal[:-1][mask[:-1]]\nX_input = full['input'][:-1][mask[:-1]].reshape(-1,1)\ny = np.array([1 if x == 1 else 0 for x in full['q'][1:][mask[:-1]]])\nsplit = int(.7 * len(y))\nm1 = LogisticRegression(max_iter=1000).fit(X_internal[:split], y[:split])\nm0 = LogisticRegression(max_iter=1000).fit(X_input[:split], y[:split])\nacc_internal = float(accuracy_score(y[split:], m1.predict(X_internal[split:])))\nacc_input = float(accuracy_score(y[split:], m0.predict(X_input[split:])))\nprint('INTERNAL_STATE_ACCURACY =', acc_internal)\nprint('INPUT_ONLY_ACCURACY =', acc_input)\nprint('SELF_STATE_INFORMATION_GAIN =', acc_internal - acc_input)\n

## 6. Visualización del recorrido\n

In [ ]:
plt.figure(figsize=(14,4))\nplt.plot(full['state'][-1500:], label='state')\nplt.plot(full['theta'][-1500:], label='+theta', alpha=.7)\nplt.plot(-full['theta'][-1500:], label='-theta', alpha=.7)\nplt.title('Estado relacional y banda crítica')\nplt.xlabel('step')\nplt.ylabel('state')\nplt.legend()\nplt.show()\n

## 7. Interpretación\n\nGuardar números, gráficos y configuración junto con cada corrida. Un resultado llamativo no es suficiente: debe sobrevivir a semillas múltiples, ablaciones y cambios de distribución.\n